# Práctica voluntariado

## Leer y unir

### 1. Leer fuentes con parámetros correctos y calcular horas totales

* Procedemos a leer los dos archivos csv, encontrándonos que las columnas fechas están siendo interpretadas como str en vez de datetime, para ello las convertirmos según los formatos existentes.

In [1]:
import json
import unicodedata
from pathlib import Path
import secrets
import pandas as pd
import hashlib
import hmac
import os 
from dotenv import load_dotenv

load_dotenv()

RAW = Path("../datos/raw")
CLEAN = Path("../datos/clean")
RAW.mkdir(parents=True, exist_ok=True)
CLEAN.mkdir(parents=True, exist_ok=True)

print(sorted(p.name for p in RAW.iterdir()))

['inscricions.csv', 'quendas.csv']


In [2]:
quendas = pd.read_csv(RAW / "quendas.csv", sep=";", decimal=",")
inscricions = pd.read_csv(RAW / "inscricions.csv")

for nome, df in [("quendas", quendas), ("inscricions", inscricions)]:
    print(f"{nome:11s} {df.shape}  columnas: {list(df.columns)}")

quendas     (704, 5)  columnas: ['data', 'quenda', 'tarefa', 'email', 'horas']
inscricions (140, 5)  columnas: ['nome', 'email', 'data_nacemento', 'concello', 'data_alta']


In [3]:
quendas.info()

<class 'pandas.DataFrame'>
RangeIndex: 704 entries, 0 to 703
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   data    704 non-null    str    
 1   quenda  704 non-null    str    
 2   tarefa  704 non-null    str    
 3   email   704 non-null    str    
 4   horas   704 non-null    float64
dtypes: float64(1), str(4)
memory usage: 27.6 KB


In [4]:
inscricions.info()

<class 'pandas.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   nome            140 non-null    str  
 1   email           140 non-null    str  
 2   data_nacemento  140 non-null    str  
 3   concello        140 non-null    str  
 4   data_alta       140 non-null    str  
dtypes: str(5)
memory usage: 5.6 KB


In [5]:
def converter_datas(serie, formatos):
    """Proba cada formato por orde e devolve as datas e os textos que non encaixaron en ningún."""
    resultado = pd.Series(pd.NaT, index=serie.index, dtype="datetime64[ns]") #serie vacía para guardar los resultados
    for formato in formatos: #intentamos convertir cada una de las fechas a formato datetime con cada uno de los formatos indicados
        intento = pd.to_datetime(serie, format=formato, errors="coerce") #si un valor no encaja con el formato, lo convierte en NaT
        resultado = resultado.fillna(intento) #guarda el intento que se ha podido convertir
    return resultado, serie[resultado.isna()]

quendas["data"], fallos = converter_datas(quendas["data"], ["%Y-%m-%d", "%d/%m/%Y", "%y%m%d"])
inscricions["data_alta"], fallos = converter_datas(inscricions["data_alta"], ["%Y-%m-%d", "%d/%m/%Y", "%y%m%d"])
inscricions["data_nacemento"], fallos = converter_datas(inscricions["data_nacemento"], ["%Y-%m-%d", "%d/%m/%Y", "%y%m%d"])

print("Sin convertir:", fallos.tolist())#imprimimos intentos que no se han podido convertir con ninguno de los formatos para revisar

Sin convertir: []


In [6]:
inscricions.info()

<class 'pandas.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   nome            140 non-null    str           
 1   email           140 non-null    str           
 2   data_nacemento  140 non-null    datetime64[ns]
 3   concello        140 non-null    str           
 4   data_alta       140 non-null    datetime64[ns]
dtypes: datetime64[ns](2), str(3)
memory usage: 5.6 KB


In [7]:
quendas.info()

<class 'pandas.DataFrame'>
RangeIndex: 704 entries, 0 to 703
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   data    704 non-null    datetime64[ns]
 1   quenda  704 non-null    str           
 2   tarefa  704 non-null    str           
 3   email   704 non-null    str           
 4   horas   704 non-null    float64       
dtypes: datetime64[ns](1), float64(1), str(3)
memory usage: 27.6 KB


* Calulamos las horas totales de la tabla quendas para poder compararla en futuros pasos

In [8]:
horastotales=quendas["horas"].sum()
horastotales

np.float64(2090.5)

### 2. Normalizar correo en las dos tablas

* Para poder normalizar la clave correo en las dúas tablas procedemos al análisis de los diferentes datos que aparecen en la tabla , esto para observar si siguen un formato establecido.
* En este primer análisis vemos que los correos presentan una estructura similar a "nombre.apellidos@correo.example", pero en muchos de estos existen espacios en blanco

In [9]:
print(sorted(quendas["email"].unique())[:12])

[' alba.cotelo96@correo.example ', ' alba.cotelo@correo.example ', ' alba.rey@correo.example ', ' ana.souto@correo.example ', ' ana.vilar@correo.example ', ' brais.cotelo@correo.example ', ' brais.dopico@correo.example ', ' brais.rey@correo.example ', ' carme.bermudez@correo.example ', ' carme.calvo27@correo.example ', ' carme.cotelo@correo.example ', ' carme.pena@correo.example ']


In [10]:
print(sorted(inscricions["email"].unique())[:12])

['alba.calvo@correo.example', 'alba.castro60@correo.example', 'alba.castro@correo.example', 'alba.cotelo59@correo.example', 'alba.cotelo96@correo.example', 'alba.cotelo@correo.example', 'alba.mourino@correo.example', 'alba.rey@correo.example', 'ana.calvo@correo.example', 'ana.lema@correo.example', 'ana.souto@correo.example', 'ana.vilar@correo.example']


* Para no dejar ninguna posibilidad procederemos a normalizar las claves , para ello quitamos los espacios en blanco , mayúsculas y tíldes

In [11]:
def normalizar(texto):
    """Minúsculas, sen tiles e sen espazos sobrantes."""
    texto = unicodedata.normalize("NFKD", str(texto).strip().lower())
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return " ".join(texto.split())

inscricions["clave_email"] = inscricions["email"].map(normalizar)
quendas["clave_email"] = quendas["email"].map(normalizar)
print("Emails distintos en inscricions:", inscricions["email"].nunique(), "→ claves:", inscricions["clave_email"].nunique())
print("Emails distintos en quendas:", quendas["email"].nunique(), "→ claves:", quendas["clave_email"].nunique())


Emails distintos en inscricions: 140 → claves: 140
Emails distintos en quendas: 305 → claves: 113


### 3. Unir quendas con inscricions
* Procedemos a realizar un merge por la nueva columna normalizada "clave_email" , en este indicamos con validate="m:1" que existen muchas quendas para una sola incricion

In [12]:
unido = quendas.merge(inscricions, on=["clave_email"], how="left", validate="m:1", indicator=True)
unido["_merge"].value_counts()

_merge
both          700
left_only       4
right_only      0
Name: count, dtype: int64

* Aunque se ha realizado el merge tenemos datos que no han conseguido juntarse , esto debido a que no tienen el mismo formato y no consiguen juntarse con su correspondiente en inscricions
    * En el caso de 'carme.calvo@coreo.example' vemos que su correspondiente en inscricion es 'carme.calvo@correo.example' ya que si tiene inscricion, no han casado porque está registrado en quendas como 'coreo' no 'correo' 
    * En el caso de 'sabelalema@correo.example', su correspodiente es 'sabela.lema@correo.example', no se ha realizado correctamente el merge ya que falta un '.' entre el nombre y el apellido
    * Pero en el caso de 'voluntario.novo@correo.example' vemos que no tiene correspondiente en inscricion , esto debido a que seguramente sea un usuario recién registrado , asi que procedemos a excluirlo

In [13]:
sin_unir = unido[unido["_merge"] == "left_only"]
sin_unir[["nome", "quenda", "clave_email","data","data_alta","data_nacemento","tarefa","horas"]]

,nome,quenda,clave_email,data,data_alta,data_nacemento,tarefa,horas
700,NaN,mañá,carme.calvo@coreo.example,2026-10-15,NaT,NaT,Reparto,3.0
701,NaN,mañá,carme.calvo@coreo.example,2026-10-15,NaT,NaT,Reparto,3.0
702,NaN,mañá,sabelalema@correo.example,2026-10-15,NaT,NaT,Reparto,3.0
703,NaN,mañá,voluntario.novo@correo.example,2026-10-15,NaT,NaT,Reparto,3.0


In [14]:
inscricions[inscricions["clave_email"].str.contains("carme.calvo", na=False)]

,nome,email,data_nacemento,concello,data_alta,clave_email
65,Carme Calvo,carme.calvo@correo.example,1957-12-10,Oleiros,2026-03-03,carme.calvo@correo.example
82,Carme Calvo,carme.calvo27@correo.example,1971-05-06,A Coruña,2024-11-30,carme.calvo27@correo.example


In [15]:
inscricions[inscricions["clave_email"].str.contains("sabela", na=False)]

,nome,email,data_nacemento,concello,data_alta,clave_email
14,Sabela Mouriño,sabela.mourino@correo.example,1978-07-02,A Coruña,2024-12-11,sabela.mourino@correo.example
52,Sabela Vilar,sabela.vilar@correo.example,2004-07-10,Cambre,2024-10-25,sabela.vilar@correo.example
58,Sabela Lema,sabela.lema@correo.example,1978-08-01,A Coruña,2024-05-17,sabela.lema@correo.example
59,Sabela Souto,sabela.souto@correo.example,1980-07-22,Culleredo,2026-04-04,sabela.souto@correo.example
72,Sabela Souto,sabela.souto72@correo.example,1976-02-27,A Coruña,2024-10-12,sabela.souto72@correo.example
81,Sabela Dopico,sabela.dopico@correo.example,2000-02-21,A Coruña,2024-06-10,sabela.dopico@correo.example
93,Sabela Calvo,sabela.calvo@correo.example,1990-04-15,A Coruña,2024-07-23,sabela.calvo@correo.example
115,Sabela Bermúdez,sabela.bermudez@correo.example,1973-11-02,Cambre,2024-07-21,sabela.bermudez@correo.example
122,Sabela Castro,sabela.castro@correo.example,1979-03-26,A Coruña,2025-09-21,sabela.castro@correo.example


In [16]:
inscricions[inscricions["clave_email"].str.contains("voluntario", na=False)]

,nome,email,data_nacemento,concello,data_alta,clave_email


* En el caso de los correos que tengan un candidato claro ,procedemos a reemplazar el formato incorrecto para poder casar con los correos en inscricion y volver a realizar un merge
* Pero si no tenemos un candidato claro , se excluye el usuario ya que nunca vamos a poder conseguir un merge correcto con ello

In [17]:
quendas.loc[quendas["clave_email"] == "carme.calvo@coreo.example","clave_email"] = "carme.calvo@correo.example"
quendas.loc[quendas["clave_email"] == "sabelalema@correo.example","clave_email"] = "sabela.lema@correo.example"

In [18]:
unido_final = quendas.merge(inscricions, on=["clave_email"], how="left", validate="m:1", indicator=True)
unido_final["_merge"].value_counts()

_merge
both          703
left_only       1
right_only      0
Name: count, dtype: int64

In [19]:
sin_unir_corregido = unido_final[unido_final["_merge"] == "left_only"]
sin_unir_corregido[["nome", "quenda", "clave_email","data","data_alta","data_nacemento","tarefa","horas"]]

,nome,quenda,clave_email,data,data_alta,data_nacemento,tarefa,horas
703,NaN,mañá,voluntario.novo@correo.example,2026-10-15,NaT,NaT,Reparto,3.0


In [20]:
print(f"Excluídas: {len(sin_unir_corregido)} , {sin_unir_corregido['horas'].sum():.2f} horas")
unido_corregido = unido_final[unido_final["_merge"] == "both"].drop(columns="_merge")

Excluídas: 1 , 3.00 horas


### 4. Comprobamos que las horas coinciden

In [21]:
total_horas= unido_corregido["horas"].sum() + sin_unir_corregido["horas"].sum() 
print(total_horas == horastotales)

True


## Proteger

### 1.  Clasificamos las diferentes columnas entre tipos de identificadores , ya sean directos , cuasi-identificadores o atributos.

In [22]:
merge_presentar=unido_corregido[["nome", "quenda", "clave_email","data","data_alta","data_nacemento","concello","tarefa","horas"]]
merge_presentar.head()

,nome,quenda,clave_email,data,data_alta,data_nacemento,concello,tarefa,horas
0,Elena Souto,mañá,elena.souto@correo.example,2026-10-02,2024-03-08,1973-07-25,Arteixo,Reparto,3.5
1,Roi Cotelo,mañá,roi.cotelo@correo.example,2026-09-06,2026-05-05,1994-08-26,A Coruña,Clasificación,2.5
2,Iria Varela,mañá,iria.varela@correo.example,2026-10-25,2024-08-16,1970-10-20,A Coruña,Clasificación,2.5
3,Elena Rial,mañá,elena.rial@correo.example,2026-10-30,2025-09-14,2000-04-29,A Coruña,Clasificación,3.0
4,Xurxo Fernández,mañá,xurxo.fernandez98@correo.example,2026-09-02,2024-06-14,1962-03-13,Cambre,Recollida en supermercados,4.0


* Tabla de clasificación :

| Tipo | Columnas | Conclusión |
|---|---|---|
| **Identificadores directos** | `nome`, `email` | Identifican por si solos: quitar ou substituír |
| **Cuasi-identificadores** | `data_nacemento`,`concello`,`data_alta` | Por separado no identifican; **combinados, si** |
| **Atributos** | `horas`, `data`,`tarefa` | Analizar |

### 2. Clave secreta
* Para poder animizar los mails vamos a proceder a generar una función pseudonimo el cual aplica una HMAC con clave secreta a nuestro email para que no pueda ser identificado

In [23]:
clave = os.getenv("HMAC_SECRET")

def pseudonimo(texto):
    return hmac.new(clave.encode(), texto.encode(), hashlib.sha256).hexdigest()[:12]
print(pseudonimo("99861168K"), pseudonimo("99861168K")) #ejemplo de como funciona

b7ff3ab5d46a b7ff3ab5d46a


### 3. Generalizar las fechas de nacimiento en tramos de edad y agrupar los concellos con menos de 5 personas en "Outros"
* En este paso procedemos a trabajar con un nuevo dataframe , así eliminado las columnas que no van a ser tratadas en este nuevo análisis

In [24]:
datos_presentar = merge_presentar.copy()

idade = 2026 - pd.to_datetime(datos_presentar["data_nacemento"]).dt.year
datos_presentar["tramo_edad"] = pd.cut(idade, bins=[0, 29, 39, 49, 59, 69, 120],
                                       labels=["<30", "30-39", "40-49", "50-59", "60-69", "70+"])

frecuencias = datos_presentar.groupby("concello")["clave_email"].nunique()

datos_presentar.loc[datos_presentar["concello"].map(frecuencias) < 5, "concello"] = "Outros"

datos_presentar["id_participante"] = datos_presentar["clave_email"].map(pseudonimo) # hmac para los mails
datos_presentar = datos_presentar[["id_participante", "tramo_edad","concello","tarefa","horas"]]
datos_presentar.head()

,id_participante,tramo_edad,concello,tarefa,horas
0,33327d1d45e0,50-59,Arteixo,Reparto,3.5
1,7e2f295512ef,30-39,A Coruña,Clasificación,2.5
2,d36a702eb035,50-59,A Coruña,Clasificación,2.5
3,08a449ea21e8,<30,A Coruña,Clasificación,3.0
4,d5ae022087de,60-69,Cambre,Recollida en supermercados,4.0


In [25]:
datos_presentar.groupby("concello")["id_participante"].nunique().sort_values()

concello
Outros        6
Cambre        8
Arteixo       9
Oleiros      10
Culleredo    11
A Coruña     66
Name: id_participante, dtype: int64

### 4. Calcular k

* Para poder calcular k agrupamos por tramo_edad y concello para poder identificar a nuestro grupo más pequeños
    * Con k nos referimos a las veces que cada combinación de cuasi-identificadores aparece por lo menos k veces

* En el caso de que tengamos que presentar fila por fila necesitaríamos hacer los siguientes pasos:
    * Mantener parametrización de email pero solo mostrando clave_email y generalización
    * Aplicar generalización a los diferentes datos tipos fecha , es decir , en vez de mostrar fecha_alta podríamos generalizarlo con un rango de meses 
    * Con data la cual se refiera a fecha de relización de la actividad , podríamos generalizar con semanas por número de semana del mes, esto para que no se llegen a perder los datos o no se lleguen a entender
    * Podría patametrizarse el nombre o directamente eliminarlo
    * En el caso del concello podríamos generalizarlo de tal forma que en vez de referirse a concellos directos ,se refiriesen a areas de la ciudad

In [26]:
grupos = datos_presentar.groupby(["tramo_edad","concello"], observed=True).size()
print("k =", grupos.min(), "· grupos con menos de 5 persoas:", (grupos < 5).sum())
grupos[grupos < 5]

k = 2 · grupos con menos de 5 persoas: 2


tramo_edad  concello
30-39       Cambre      2
70+         Cambre      4
dtype: int64

* Para poder limpiar estos grupos , procedemos a modificar los tramos de edad de 70+ 
    * Reemplazamos este dato por unso rangos más diversificados para que así sea más dificil de localizar

In [27]:
datos_presentar["tramo_edad"] = datos_presentar["tramo_edad"].astype(str).replace({"60-69": "60+", "70+": "60+"})
grupos = datos_presentar.groupby(["tramo_edad", "concello"]).size()
print("k =", grupos.min())
grupos[grupos < 5]

k = 2


tramo_edad  concello
30-39       Cambre      2
dtype: int64

* Pero en cierto momento si siguen existiendo grupos que no cumplen nuestro criterio los excluimos 

In [28]:
K = 5
tamano = datos_presentar.groupby(["tramo_edad","concello"])["id_participante"].transform("size")
print("Personas suprimidas:", (tamano < K).sum())
datos_presentar = datos_presentar[tamano >= K]
print("k =", datos_presentar.groupby(["tramo_edad", "concello"]).size().min())

Personas suprimidas: 2
k = 5


## Publicar y CSV

* Para publicar la tabla que muestros las horas por tramo de edad y tarefa procedemos a realizar un pivotaje en el que sumamos las horas y convertimos en columnas a las diferentes tarefas
    * Realizamos un fltro adicional , que consiste en reemplazar los datos que tengan menos de 5 personas distintas por grupo de tramo de edad

In [29]:
personas = datos_presentar.pivot_table(index="tramo_edad",columns="tarefa",values="id_participante",aggfunc="nunique",fill_value=0)
personas

tarefa,Clasificación,Recollida en supermercados,Reparto
tramo_edad,,,
30-39,18,19,19
40-49,20,20,21
50-59,21,23,21
60+,19,18,18
<30,15,17,15


In [30]:
informe = datos_presentar.pivot_table(index="tramo_edad",columns="tarefa",values="horas",aggfunc="sum",fill_value=0)
informe
publicable = informe.astype(str).where(personas >= 5, "<5") #si la condición es true conserva el valor original , si no lo sustituye por "<5"
publicable

tarefa,Clasificación,Recollida en supermercados,Reparto
tramo_edad,,,
30-39,137.0,144.5,134.5
40-49,178.5,132.0,145.0
50-59,160.5,145.5,158.5
60+,133.5,162.0,143.5
<30,82.0,118.5,105.5


In [31]:
# Guardar datos limpios en csv
publicable.to_csv("../datos/clean/informe_limpio.csv", index=True, sep=";", encoding="utf-8")

## Uso de IA
* Se utilizó chatgpt para realizar las consultas con IA
* Generar código de ejemplo para poder realizar el filtro de la tabla a publicar 
* Generar código de ejemplo para poder realizar el reemplazo de Outros por los valores indicados